# UTAC-ODE vs. CMIP6/ERA5 Benchmark

**Hypothese**: Das UTAC-Logistic-Modell, angetrieben durch CREP-Gamma aus ERA5-Temperaturanomalien,
prognostiziert den Eisvolumenverlust im Testzeitraum **statistisch signifikant besser** als eine
lineare Baseline.

| | Trainingsperiode | Testperiode |
|---|---|---|
| Jahre | 1940 – 2010 | 2011 – 2023 |
| Punkte | 71 | 13 |

**Daten**: ERA5-Reanalyse (Proxy für CMIP6 ScenarioMIP)  
**Reproduzierbarkeit**: Seed 42, deterministisch, < 5 Sekunden Laufzeit  
**Methode**: UTAC-Logistic ODE + CREP-Gamma aus normierter Temperaturanomalie

In [ ]:
import sys
import csv
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

repo_root = pathlib.Path("__file__").resolve().parent.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from genesis_os.core.crep import CREPEvaluator
from genesis_os.runtime.utac import UTACLogistic

np.random.seed(42)
print("genesis_os importiert ✓ | numpy", np.__version__)

In [ ]:
data_path = repo_root / "data" / "era5_kipppunkte.csv"

years, temp_anomaly, ice_volume = [], [], []
with open(data_path, newline="") as f:
    for row in csv.DictReader(f):
        years.append(int(row["year"]))
        temp_anomaly.append(float(row["temp_anomaly"]))
        ice_volume.append(float(row["ice_volume"]))

years    = np.array(years)
temp_an  = np.array(temp_anomaly)
ice_vol  = np.array(ice_volume)

# Normalize temperature anomaly to [0, 1] as entropy proxy
t_min, t_max = temp_an.min(), temp_an.max()
entropy_sig = (temp_an - t_min) / (t_max - t_min)

# Train / test split
SPLIT_YEAR = 2011
split = int(np.where(years >= SPLIT_YEAR)[0][0])

print(f"Datenpunkte: {len(years)}  ({years[0]}–{years[-1]})")
print(f"Train: {years[0]}–{years[split-1]} ({split} Punkte)")
print(f"Test:  {years[split]}–{years[-1]} ({len(years)-split} Punkte)")

## Lineare Baseline (auf Trainingsdaten kalibriert)

Lineare Regression der Temperaturanomalie auf das Eisvolumen, kalibriert auf 1940–2010.
Vorhersage für den Testzeitraum 2011–2023.

In [ ]:
coeffs = np.polyfit(temp_an[:split], ice_vol[:split], deg=1)
ice_lin_train = np.polyval(coeffs, temp_an[:split])
ice_lin_test  = np.polyval(coeffs, temp_an[split:])

## UTAC-ODE Simulation

CREP-Gamma wird aus der normierten Temperaturanomalie abgeleitet:

$$
C = 0.80 \quad
R(t) = 0.2 + 0.6\,\epsilon(t) \quad
E(t) = 0.1 + 0.7\,\epsilon(t) \quad
P = 0.5
$$

$$
\Gamma(t) = \frac{C\cdot R(t) + E(t)\cdot P}{2}\;
\exp\!\left(-\frac{(1-C)^2}{2\,\sigma_C^2}\right)
$$

Das UTAC-Modell integriert dann:

$$
\frac{dH}{dt} = r\,H\!\left(1 - \frac{H}{K}\right)\tanh(\sigma\,\Gamma)
$$

mit $r=0.12$, $K=1.0$, $\sigma=2.2$, $H_0=0.15$.

In [ ]:
evaluator = CREPEvaluator(sigma_c=0.3)
gamma_series = []
for i, e in enumerate(entropy_sig):
    score = evaluator.evaluate({
        "coherence": 0.80,
        "resonance": float(0.2 + 0.6 * e),
        "emergence": float(0.1 + 0.7 * e),
        "poetics":   0.5,
        "cycle":     i,
    })
    gamma_series.append(score.gamma)

gamma_arr = np.array(gamma_series)

utac = UTACLogistic(r=0.12, K=1.0, sigma=2.2, dt=1.0)
H = np.zeros(len(years))
H[0] = 0.15
for i in range(1, len(years)):
    H[i] = utac.step(H[i - 1], gamma_arr[i - 1])

ice_max, ice_min = ice_vol.max(), ice_vol.min()
ice_utac = ice_max - H * (ice_max - ice_min)

print(f"H-Verlauf: {H[0]:.4f} → {H[split-1]:.4f} (2010) → {H[-1]:.4f} (2023)")
print(f"Gamma-Bereich: {gamma_arr.min():.4f} … {gamma_arr.max():.4f}")

In [ ]:
def rmse(pred: np.ndarray, obs: np.ndarray) -> float:
    return float(np.sqrt(np.mean((pred - obs) ** 2)))

def r2(pred: np.ndarray, obs: np.ndarray) -> float:
    ss_res = np.sum((obs - pred) ** 2)
    ss_tot = np.sum((obs - obs.mean()) ** 2)
    return float(1.0 - ss_res / ss_tot) if ss_tot > 0 else 0.0

# Training metrics
rmse_utac_tr  = rmse(ice_utac[:split],  ice_vol[:split])
rmse_lin_tr   = rmse(ice_lin_train,     ice_vol[:split])
r2_utac_tr    = r2(ice_utac[:split],    ice_vol[:split])
r2_lin_tr     = r2(ice_lin_train,       ice_vol[:split])

# Test metrics
rmse_utac_te  = rmse(ice_utac[split:],  ice_vol[split:])
rmse_lin_te   = rmse(ice_lin_test,      ice_vol[split:])
r2_utac_te    = r2(ice_utac[split:],    ice_vol[split:])
r2_lin_te     = r2(ice_lin_test,        ice_vol[split:])

print(f"{'':22} {'RMSE':>8} {'R²':>8}")
print("─" * 40)
print(f"{'Train  UTAC-ODE':<22} {rmse_utac_tr:>8.4f} {r2_utac_tr:>8.4f}")
print(f"{'Train  Linear':<22} {rmse_lin_tr:>8.4f} {r2_lin_tr:>8.4f}")
print("─" * 40)
print(f"{'Test   UTAC-ODE':<22} {rmse_utac_te:>8.4f} {r2_utac_te:>8.4f}  ← Benchmark")
print(f"{'Test   Linear':<22} {rmse_lin_te:>8.4f} {r2_lin_te:>8.4f}  ← Baseline")
imp = (rmse_lin_te - rmse_utac_te) / rmse_lin_te * 100
print(f"\nRMSE-Verbesserung (Test): {imp:+.1f} %")

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.30)

# ── Panel 1: Zeitreihe mit Train/Test-Split ──────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.axvspan(years[split], years[-1] + 0.5, alpha=0.07, color="gold", label="Testperiode")
ax1.plot(years, ice_vol, color="steelblue", lw=2.0,
         label="ERA5/CMIP6 (beobachtet)")
ax1.plot(years, ice_utac, color="darkorchid", lw=2.0, linestyle="--",
         label=f"UTAC-ODE  test RMSE={rmse_utac_te:.3f} | R²={r2_utac_te:.3f}")
ax1.plot(years[:split], ice_lin_train, color="gray", lw=1.5, linestyle=":",
         label=f"Linear (Train)")
ax1.plot(years[split:], ice_lin_test, color="darkorange", lw=1.5, linestyle=":",
         label=f"Linear (Test)  RMSE={rmse_lin_te:.3f} | R²={r2_lin_te:.3f}")

# Kipppunkt-Markierung
grad = np.gradient(ice_utac, years)
kipp_idx = int(np.argmin(grad))
ax1.axvline(years[kipp_idx], color="red", linestyle=":", alpha=0.8,
            label=f"Kipppunkt ~ {years[kipp_idx]}")

ax1.set_title("Eisvolumen-Zeitreihe: UTAC-ODE vs. ERA5/CMIP6  (Train 1940–2010 | Test 2011–2023)",
              fontsize=12)
ax1.set_xlabel("Jahr")
ax1.set_ylabel("Eisvolumen (normiert)")
ax1.legend(fontsize=8.5)
ax1.grid(True, alpha=0.3)

# ── Panel 2: CREP-Gamma-Verlauf ──────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(years, gamma_arr, color="teal", lw=2.0, label="CREP-Gamma Γ(t)")
ax2.axhline(0.6, color="red", linestyle="--", alpha=0.7, label="Phase-Transition-Schwelle")
ax2.fill_between(years, gamma_arr, 0.6,
                 where=(gamma_arr >= 0.6), alpha=0.15, color="red")
ax2.axvline(SPLIT_YEAR, color="gold", linestyle="-", lw=1.5, alpha=0.8)
ax2.set_title("CREP-Kopplungsterm Γ(t)", fontsize=11)
ax2.set_xlabel("Jahr")
ax2.set_ylabel("Γ")
ax2.legend(fontsize=8.5)
ax2.grid(True, alpha=0.3)

# ── Panel 3: Phasenraum T-Anomalie vs Eisvolumen ─────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
sc = ax3.scatter(temp_an[:split], ice_vol[:split], c="steelblue", s=25,
                 label="Train (beobachtet)", zorder=3, alpha=0.8)
ax3.scatter(temp_an[split:], ice_vol[split:], c="gold", s=40,
            edgecolors="k", linewidths=0.5,
            label="Test (beobachtet)", zorder=4)
ta_sorted = np.sort(temp_an)
ax3.plot(ta_sorted, np.polyval(coeffs, ta_sorted), color="darkorange",
         lw=1.5, linestyle=":", label="Linear")
idx_sort = np.argsort(temp_an)
ax3.plot(temp_an[idx_sort], ice_utac[idx_sort], color="darkorchid",
         lw=1.8, linestyle="--", label="UTAC-ODE")
ax3.set_title("Phasenraum: T-Anomalie vs. Eisvolumen", fontsize=11)
ax3.set_xlabel("Temperaturanomalie (°C)")
ax3.set_ylabel("Eisvolumen (normiert)")
ax3.legend(fontsize=8.5)
ax3.grid(True, alpha=0.3)

fig.suptitle(
    "GenesisAeon – UTAC-ODE vs. CMIP6/ERA5 Kipppunkt-Benchmark  (Seed 42, deterministisch)",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Ergebniszusammenfassung ───────────────────────────────────────────────
imp_rmse = (rmse_lin_te - rmse_utac_te) / rmse_lin_te * 100
imp_r2   = r2_utac_te - r2_lin_te

print("=" * 54)
print("UTAC-ODE vs. CMIP6/ERA5 – Benchmark-Ergebnis")
print("=" * 54)
print(f"Test-RMSE  UTAC    : {rmse_utac_te:.4f}")
print(f"Test-RMSE  Linear  : {rmse_lin_te:.4f}")
print(f"RMSE-Verbesserung  : {imp_rmse:+.1f} %")
print(f"R²-Verbesserung    : {imp_r2:+.4f}")
print(f"Kipppunkt detektiert (UTAC) : {years[kipp_idx]}")
print(f"Gamma bei Kipppunkt         : {gamma_arr[kipp_idx]:.4f}")
print()
print("Parameter: UTACLogistic(r=0.12, K=1.0, sigma=2.2, dt=1.0), H0=0.15")
print("CREP:      C=0.80  R=0.2+0.6ε  E=0.1+0.7ε  P=0.5")
print("Zenodo-DOI (geplant): 10.5281/zenodo.genesis-os")

## Reproduktionsanleitung

```bash
# Im genesis-os Verzeichnis
pip install -e .
jupyter lab notebooks/benchmark_utac_vs_cmip6.ipynb
# Run → Run All Cells
```

**Erwartetes Ergebnis**:
- UTAC-ODE RMSE (Test 2011–2023) kleiner als lineare Baseline
- Kipppunkt zwischen 1995 und 2010 detektiert
- Gamma-Kurve überschreitet Phase-Transition-Schwelle (0.6) im Testzeitraum

**Weiter**: Whitepaper-Abschnitt 4.2 (Klimavalidierung) + arXiv-Upload.